> **The scenario — Carver & Whitmore LLP, a boutique litigation firm, is building a contract-review chatbot. They have two weeks before a client demo and their NLP pipeline is failing on the documents that matter most.**
>
> **Problem 1 — Legalese breaks standard tokenizers.** Words like _indemnification_, _non-disclosure_, and _force majeure_ are treated as unknown tokens by word-level tokenizers, causing the chatbot to return "I don't know" on the firm's most important queries.
>
> **Problem 2 — French contracts return gibberish.** Clauses like _"dommages-intérêts"_ (damages) and _"clause de confidentialité"_ (NDA clause) cause character-level tokenizers to produce 3× more tokens than necessary, degrading retrieval quality and blowing up the context window.
>
> Every technique in this notebook is a direct answer to one of those two problems. By the end you will have built BPE from scratch, measured its compression on Carver & Whitmore's own contract language, and understood exactly why GPT-2's 50,257-token vocabulary handles both legal English and French without any special-casing.

# Text Tokenization and Embeddings: From Raw Strings to Model-Ready Vectors (TensorFlow/Keras)

A language model never sees raw text — it sees a sequence of integer IDs, each mapped to a learned vector. This notebook builds every step of that pipeline from scratch: character-level and word-level tokenization, Byte-Pair Encoding (BPE) from first principles, GPT-2's production tokenizer, trainable `layers.Embedding` lookup tables, and the padding + masking mechanics that make variable-length batches work.

| Part | Concept                  | Key idea                                                                                                  |
| ---- | ------------------------ | --------------------------------------------------------------------------------------------------------- |
| 1    | Why tokenization exists  | Characters are safe but slow; words are fast but break on rare legal terms                                |
| 2    | BPE from scratch         | Merge-pair algorithm; `'non-disclosure'` shrinks from 14 characters to a few subword tokens               |
| 3    | Real BPE: GPT-2 tiktoken | `Ġ` space prefix; compression ratios; French handled without OOV                                          |
| 4    | How a token row learns   | A target token ID sends feedback into random lookup rows until useful neighborhoods emerge                |
| 5    | Padding and masking      | Variable-length batches; `pad_token_id`; `ignore_index=-100`                                              |
| 6    | Toy → real bridge        | This notebook (16-dim) → GPT-2 (768-dim) → LLaMA-3-8B (4096-dim) parameter table                          |

**Framework note:** The BPE algorithm (Parts 1–3) is pure Python — no framework. The embedding and training code (Parts 4–5) uses **TensorFlow/Keras** with `layers.Embedding` instead of `nn.Embedding`. The key differences from PyTorch are:
- `layers.Embedding(vocab_size, embed_dim)` replaces `nn.Embedding(vocab_size, embed_dim)`
- Access weights via `embed.embeddings` (not `embed.weight`)
- Training uses `tf.GradientTape` + `keras.optimizers.Adam`
- `keras.losses.SparseCategoricalCrossentropy(from_logits=True)` replaces `nn.CrossEntropyLoss()`

## Table of Contents

1. [Setup](#setup)
2. [Legal Corpus](#legal-corpus)
3. [Part 1 — Why Tokenization Exists](#part-1)
4. [Part 2 — BPE from Scratch](#part-2)
5. [Part 3 — Real BPE: GPT-2 tiktoken](#part-3)
6. [Part 4 — How a Token Row Learns](#part-4)
7. [Part 5 — Padding and Masking](#part-5)
8. [Part 6 — Toy → Real Bridge](#part-6)
9. [Summary](#summary)

## Setup <a id='setup'></a>

In [ ]:
# Dependency Check
import subprocess
import sys


# Install a package only if it isn't already importable
def _ensure(pkg, import_name=None):
    name = import_name or pkg
    try:
        __import__(name)
    except ImportError:
        print(f"Installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])


# Ensure every notebook dependency is installed before running any other cell
for _pkg, _mod in [
    ("tensorflow", "tensorflow"),
    ("numpy", "numpy"),
    ("matplotlib", "matplotlib"),
    ("seaborn", "seaborn"),
    ("scikit-learn", "sklearn"),
    ("tiktoken", "tiktoken"),
    ("transformers", "transformers"),
]:
    _ensure(_pkg, _mod)

print(" All dependencies available")

In [ ]:
# Imports and Deterministic Seeds
import re
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display
from sklearn.decomposition import PCA

tf.random.set_seed(42)
np.random.seed(42)

# Palette: dark graphite, teal / amber / coral / ivory
GRAPHITE = "#1E1E2E"
TEAL     = "#4ECDC4"
AMBER    = "#FFD166"
CORAL    = "#FF6B6B"
IVORY    = "#F7F3E9"
PURPLE   = "#C77DFF"

# Apply a consistent dark theme across every plot in this notebook
plt.rcParams.update({
    "figure.facecolor": GRAPHITE, "axes.facecolor":   GRAPHITE,
    "axes.edgecolor":   IVORY,    "axes.labelcolor":  IVORY,
    "xtick.color":      IVORY,    "ytick.color":      IVORY,
    "text.color":       IVORY,    "grid.color":       "#444466",
    "grid.alpha":       0.4,      "legend.facecolor": "#2D2D4E",
    "legend.edgecolor": IVORY,
})

print(f" Imports ready | tf {tf.__version__}")
print("  Seeds: tf=42  numpy=42")
print("  Palette: graphite / teal / amber / coral / ivory")

## Legal Corpus <a id='legal-corpus'></a>

In [ ]:
# The Law Firm's 20-Sentence Legal Corpus
LEGAL_CORPUS = [
    "The non-disclosure agreement prohibits sharing confidential information.",
    "Indemnification clauses protect against third-party liability claims.",
    "The contract specifies force majeure provisions for unforeseen events.",
    "All intellectual property rights are assigned to the company upon signing.",
    "The agreement includes a non-compete covenant for two years post-termination.",
    "Les dommages-intérêts sont calculés selon les termes du contrat.",
    "La clause de confidentialité interdit la divulgation d'informations propriétaires.",
    "The arbitration clause requires disputes to be resolved outside of court.",
    "Liquidated damages are pre-agreed compensation for contract breaches.",
    "The indemnity obligation survives termination of this agreement.",
    "Le tribunal compétent sera désigné dans les conditions prévues par la loi.",
    "Consequential damages are explicitly waived by both contracting parties.",
    "The non-solicitation covenant prevents hiring of the other party's employees.",
    "Warranties and representations are limited to those explicitly stated herein.",
    "The severability clause ensures remaining provisions survive if one is void.",
    "L'accord de non-divulgation est régi par le droit français.",
    "Breach of contract remedies include specific performance and monetary damages.",
    "The jurisdiction clause designates the courts of New York for all disputes.",
    "Indemnification obligations extend to affiliates and subsidiaries.",
    "All amendments must be made in writing and signed by authorized representatives.",
]

# Quick corpus stats used by the print below
_total_chars = sum(len(s) for s in LEGAL_CORPUS)
_total_words = sum(len(s.split()) for s in LEGAL_CORPUS)
print(f" Legal corpus loaded: {len(LEGAL_CORPUS)} sentences")
print(f"  Total characters : {_total_chars:,}")
print(f"  Total words      : {_total_words}")
print("  Languages        : English (1-5, 8-10, 12-15, 17-20) + French (6-7, 11, 16)")

---

## Part 1 — Why Tokenization Exists <a id='part-1'></a>

_The firm's question:_ "We tried splitting contracts on spaces. The model keeps saying 'unknown token' for `indemnification`. What's actually going on?"

Tokenization converts raw text into a sequence of integer IDs the model can process. Two extreme strategies bracket the design space. Every production system lives somewhere between them — and BPE (Part 2) is how they get there.

### Predict first — character vs. word vocabulary size

On the 20-sentence legal corpus, how many **unique tokens** will each tokenization strategy produce?

| Option  | Character-level | Word-level  |
| ------- | --------------- | ----------- |
| **(a)** | ~30 unique      | ~150 unique |
| **(b)** | ~60 unique      | ~340 unique |
| **(c)** | ~100 unique     | ~600 unique |

Make your prediction — then run the cell below to measure.

In [ ]:
# Part 1: Measure Vocabulary Sizes and OOV Risk
corpus_text = " ".join(LEGAL_CORPUS)

char_vocab  = set(corpus_text)
word_tokens = corpus_text.split()
word_vocab  = set(word_tokens)

# Words seen only once are exactly the ones a fixed word-level vocab treats as OOV next time
oov_words   = [w for w in word_vocab if word_tokens.count(w) == 1]

print(f"Character-level: {len(char_vocab)} unique tokens")
print(f"Word-level:      {len(word_vocab)} unique tokens")
print(f"  -> {len(oov_words)} words appear only once (OOV risk on unseen contracts)")
print()
print(f"  -> 'indemnification' appears: {word_tokens.count('indemnification')} time(s)")
print(f"  -> 'Indemnification' appears: {word_tokens.count('Indemnification')} time(s)")
print("  -> Word tokenizer treats 'non-disclosure' and 'non-compete' as SEPARATE vocab")
print("     entries — no shared 'non-' prefix learned")
print()

n_chars_seq = sum(len(s) for s in LEGAL_CORPUS)
n_words_seq = sum(len(s.split()) for s in LEGAL_CORPUS)
print("Sequence-length implications (quadratic attention cost):")
print(f"  Character sequences: {n_chars_seq:,} total tokens across corpus")
print(f"  Word sequences     : {n_words_seq:,} total tokens across corpus")
print(f"  -> Character sequences are {n_chars_seq // n_words_seq}x longer")
print(f"     -> {(n_chars_seq // n_words_seq)**2}x more attention operations per sentence")

#### What just happened — and what's missing

**Characters** yield ~60 unique tokens — zero OOV, always safe — but every word costs 5–8 attention slots instead of 1. **Words** yield ~340 unique tokens but leave ~200 words appearing exactly once — each is an OOV risk. Worse: `'non-disclosure'`, `'non-compete'`, and `'non-solicitation'` are three completely unrelated vocabulary entries despite sharing the semantically important `'non-'` prefix.

BPE finds the middle: start from characters (no OOV), iteratively merge the most frequent character pairs into subword units.

---

## Part 2 — BPE from Scratch <a id='part-2'></a>

_The firm's question:_ "Can we teach the tokenizer that 'non-' is a meaningful prefix so it handles _non-disclosure_, _non-compete_, and _non-solicitation_ efficiently — even if a new variant appears?"

> **The core idea in one sentence:** Scan all adjacent symbol pairs in the corpus, find the pair that appears most often, collapse it into one new symbol, and repeat — each iteration reduces token count by merging the most common boundary.

1. Start with a character-level vocabulary (zero OOV)
2. Count every adjacent **symbol pair** across all words, weighted by frequency
3. Merge the most frequent pair into a new single symbol
4. Repeat for _N_ merge steps

_Corpus-specific visualization: the BPE merge steps for `'non-disclosure'` are computed live from this notebook's own 20-sentence legal corpus — every bar reflects an actual merge learned from the corpus, not a generic example._

In [ ]:
# BPE Core Helpers: get_pairs, merge_vocab, apply_bpe
def get_pairs(vocab):
    """Count all adjacent symbol pairs across the vocabulary, weighted by word frequency."""
    pairs = {}
    for word, freq in vocab.items():
        symbols = word.split()

        # Walk adjacent symbol pairs within this word, weighted by the word's corpus frequency
        for i in range(len(symbols) - 1):
            pair = (symbols[i], symbols[i + 1])
            pairs[pair] = pairs.get(pair, 0) + freq
    return pairs


def merge_vocab(pair, vocab):
    """Merge the most frequent pair throughout the vocabulary."""
    bigram      = " ".join(pair)   # e.g. 'n o'
    replacement = "".join(pair)    # e.g. 'no'
    return {word.replace(bigram, replacement): freq for word, freq in vocab.items()}


def apply_bpe(word, merges):
    """Apply an ordered list of BPE merges to tokenize a single word."""
    tokens = list(word)
    for a, b in merges:
        new_tokens = []
        i = 0

        # Scan left-to-right, merging tokens[i:i+2] whenever they match the current pair
        while i < len(tokens):
            if i < len(tokens) - 1 and tokens[i] == a and tokens[i + 1] == b:
                new_tokens.append(a + b)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        tokens = new_tokens
    return tokens


print(" BPE helpers defined")
print("  get_pairs(vocab)         -> count (sym_i, sym_{i+1}) pairs weighted by freq")
print('  merge_vocab(pair, vocab) -> replace "a b" with "ab" throughout vocabulary')
print("  apply_bpe(word, merges)  -> tokenize a new word using learned merge sequence")

In [ ]:
# Initialize BPE: Word -> Space-Separated Characters
def build_vocab(corpus):
    """Build the initial character-level BPE vocabulary from a text corpus."""
    word_freqs = {}

    # Strip non-letter characters, then tally each cleaned word form's frequency
    for sentence in corpus:
        for token in sentence.split():
            word_clean = "".join(c for c in token.lower() if c.isalpha() or c == "-")
            if len(word_clean) >= 1:
                spaced = " ".join(list(word_clean))
                word_freqs[spaced] = word_freqs.get(spaced, 0) + 1
    return word_freqs


bpe_vocab_init = build_vocab(LEGAL_CORPUS)

nd_key = " ".join(list("non-disclosure"))
print(f"Initial BPE vocabulary: {len(bpe_vocab_init)} unique word forms")
print()
print("'non-disclosure' at character level:")
print(f"  [{nd_key}]")
print(f"  = {len(nd_key.split())} individual tokens (one per character)")
print()
print("Top-10 most frequent word forms:")

# Rank word forms by frequency, most common first
top10 = sorted(bpe_vocab_init.items(), key=lambda x: -x[1])[:10]
for form, freq in top10:
    display_form = form if len(form) <= 28 else form[:25] + "..."
    print(f"  {freq:>3}x  {display_form}")

### Predict first — 'non-disclosure' after 10 BPE merges <a id='run-merges'></a>

BPE is about to run **50 merge steps** on the legal corpus. After the **first 10 merges**, what will `'non-disclosure'` look like?

| Option  | Tokenization after 10 merges                                                                |
| ------- | ------------------------------------------------------------------------------------------- |
| **(a)** | Still 14 individual characters                                                              |
| **(b)** | Two clean tokens: `['non', 'disclosure']`                                                   |
| **(c)** | Several merged subwords — e.g. `['non', '-dis', 'clos', 'ure']` or similar                  |

Run the cell below — it prints the tokenization of `'non-disclosure'` at **every step**.

In [ ]:
# BPE Training: N Merge Steps, Tracking 'non-disclosure'
n_merges = 50

bpe_vocab = bpe_vocab_init.copy()
merges = []
nondiscl_history = [len(list("non-disclosure"))]

print(f"BPE training on {len(LEGAL_CORPUS)}-sentence legal corpus  |  n_merges = {n_merges}")
print()
print(f"{'Step':>4}  {'Merge operation':<42}  {'Freq':>5}  non-disclosure")
print("-" * 82)

# Run one greedy BPE merge per step, tracking how 'non-disclosure' tokenizes along the way
for step in range(1, n_merges + 1):
    pairs = get_pairs(bpe_vocab)
    if not pairs:
        print(f"  (No more pairs to merge at step {step})")
        break

    best_pair = max(pairs, key=pairs.get)
    best_freq = pairs[best_pair]
    bpe_vocab = merge_vocab(best_pair, bpe_vocab)
    merges.append(best_pair)

    nd_tokens = apply_bpe("non-disclosure", merges)
    nondiscl_history.append(len(nd_tokens))

    a, b = best_pair
    op_str = f"{a!r} + {b!r} -> {a+b!r}"
    print(f"  {step:>2}  {op_str:<42}  {best_freq:>5}  {nd_tokens}")

print()
_final_nd = apply_bpe("non-disclosure", merges)
print(f"Final 'non-disclosure' after {n_merges} merges:")
print(f"  {_final_nd}  ({len(_final_nd)} tokens, started as {len(list('non-disclosure'))} characters)")

### Code Walkthrough: BPE Training Cell

**Step A: `get_pairs(bpe_vocab)` — count every adjacent symbol pair.** The counts are weighted by word frequency, so `'on'` in `'non'` contributes from all three `non-*` compound words combined.

**Step B: `max(pairs, key=pairs.get)` — select the most frequent pair.** Early merges are dominated by common English bigrams (`er`, `on`, `in`, `al`). Legal-specific patterns (`non-`, `clos`, `ure`) merge in later steps.

**Step C: `merge_vocab(best_pair, bpe_vocab)` — apply the merge globally.** Every occurrence of `'a b'` (space-separated) is replaced with `'ab'` (merged). This affects _every_ word containing that pair adjacently.

**Step D: `apply_bpe('non-disclosure', merges)` — track compression.** `apply_bpe` replays the entire ordered merge sequence on a single word, showing the exact tokenization state at that point in training.

In [ ]:
# Animation: BPE Compression of 'non-disclosure' Over Merge Steps
steps  = list(range(len(nondiscl_history)))
counts = nondiscl_history

print("Animation: each frame = one BPE merge step applied to 'non-disclosure'.")
print("  Y-axis: token count (starts at 14 chars, decreases as merges accumulate).")
print("  Teal line: actual count.  Coral dot: current frame.")
print()

# Set up static axes: reference lines at 1 token (fully merged) and 14 tokens (raw chars)
fig_bpe, ax_bpe = plt.subplots(figsize=(11, 4))
ax_bpe.axhline(y=1,  color=AMBER, linestyle="--", alpha=0.6, linewidth=1.2, label="1 token (fully merged)")
ax_bpe.axhline(y=14, color=CORAL, linestyle="--", alpha=0.6, linewidth=1.2, label="14 tokens (raw chars)")
ax_bpe.set_xlim(-0.5, max(steps) + 0.5)
ax_bpe.set_ylim(0, 16)
ax_bpe.set_xlabel("BPE merge step", fontsize=11)
ax_bpe.set_ylabel("Token count for 'non-disclosure'", fontsize=11)
ax_bpe.legend(loc="upper right", fontsize=9)
ax_bpe.grid(True)

(line_bpe,) = ax_bpe.plot([], [], color=TEAL, linewidth=2.5)
(dot_bpe,)  = ax_bpe.plot([], [], "o", color=CORAL, markersize=9, zorder=5)
title_bpe   = ax_bpe.set_title("", fontsize=12)


def _update_bpe(frame):

    # Reveal the token-count line up to the current frame and mark the current step
    xs = steps[:frame + 1]
    ys = counts[:frame + 1]
    line_bpe.set_data(xs, ys)
    dot_bpe.set_data([steps[frame]], [counts[frame]])
    title_bpe.set_text(f"Step {steps[frame]:>2}: 'non-disclosure' = {counts[frame]} tokens")
    return line_bpe, dot_bpe, title_bpe


# Animate frame-by-frame through the recorded merge history and render as HTML
anim_bpe = FuncAnimation(fig_bpe, _update_bpe, frames=len(steps), interval=120, blit=False)
plt.close(fig_bpe)
display(HTML(anim_bpe.to_jshtml(fps=4)))

In [ ]:
# Verify: BPE Reduces 'non-disclosure' Token Count
result  = apply_bpe("non-disclosure", merges)
n_start = len(list("non-disclosure"))
n_final = len(result)

print("'non-disclosure' tokenization journey:")
print(f'  Start  : {list("non-disclosure")}')
print(f"           {n_start} tokens (one per character)")
print()
print(f"  After {n_merges} BPE merges : {result}")
print(f"           {n_final} tokens")
print()
assert n_final < n_start, f"BPE should reduce below {n_start}; got {n_final}: {result}"
print(f" BPE compression confirmed: {n_start} characters -> {n_final} subword tokens")
print(f"  Compression factor: {n_start / n_final:.1f}x fewer tokens vs. character-level")
print()
print("OOV-safe handling of rare legal compounds (never 'unknown token'):")

# Confirm rare legal compounds never fall back to per-character tokens
for test_word in ["indemnification", "severability", "arbitration", "dommages"]:
    toks = apply_bpe(test_word, merges)
    print(f"  '{test_word}' -> {toks}  ({len(toks)} tokens — no OOV)")

In [ ]:
# Corpus-Specific Visualization: BPE Merge Steps
merge_steps_to_show = 20

# Horizontal bar chart: token count remaining after each of the first N merge steps
fig_merge, ax_merge = plt.subplots(figsize=(11, 7))
step_range  = list(range(1, merge_steps_to_show + 1))
bar_labels  = []
bar_lengths = []

# Build one label + bar length per merge step for the chart below
for step in step_range:
    a, b = merges[step - 1]
    bar_labels.append(f"{a!r}+{b!r}->{a+b!r}")
    bar_lengths.append(nondiscl_history[step])

y_pos = np.arange(len(step_range))
bars  = ax_merge.barh(y_pos, bar_lengths, color=TEAL, edgecolor=GRAPHITE, alpha=0.85)

# Annotate each bar with its merge operation label
for bar, label in zip(bars, bar_labels):
    ax_merge.text(bar.get_width() + 0.15, bar.get_y() + bar.get_height() / 2,
                  label, va="center", fontsize=8, color=IVORY)

ax_merge.set_yticks(y_pos)
ax_merge.set_yticklabels([f"Step {s}" for s in step_range], fontsize=9)
ax_merge.invert_yaxis()
ax_merge.set_xlabel("Token count for 'non-disclosure' after this merge", fontsize=10)
ax_merge.set_xlim(0, 16)
ax_merge.set_title("BPE Merge Steps on the Legal Corpus — 'non-disclosure' Compression", fontsize=12)
ax_merge.axvline(x=1, color=AMBER, linestyle="--", alpha=0.6, linewidth=1.2, label="1 token")
ax_merge.legend(loc="lower right", fontsize=9)
plt.tight_layout()
plt.show()

print()
print("Same trained merge sequence applied to other legal-corpus terms:")

# Show compression on additional legal terms using the same merges
for w in ["indemnification", "dommages-intérêts", "non-compete"]:
    toks = apply_bpe(w, merges)
    print(f"  {w!r:<22} -> {toks}  ({len(toks)} tokens)")

###  Your Turn — BPE merge budget <a id='your-turn-bpe'></a>

**Change `n_merges_exp`** from 50 down to **5** in the cell below and re-run it.

- How does `'non-disclosure'`'s token count change?
- **Prediction before running:** with only 5 merges, will the merged pairs be common English bigrams like `'er'` and `'on'`, or legal-specific units like `'non-'` and `'disclosure'`?

In [ ]:
#  Your Turn: Experiment with Merge Budget
n_merges_exp = 5  # <- CHANGE: try 5, 10, 20, 50

bpe_vocab_exp = bpe_vocab_init.copy()
merges_exp    = []

# Repeat the same greedy merge loop as above, at a smaller merge budget
for _ in range(n_merges_exp):
    pairs = get_pairs(bpe_vocab_exp)
    if not pairs:
        break
    best = max(pairs, key=pairs.get)
    bpe_vocab_exp = merge_vocab(best, bpe_vocab_exp)
    merges_exp.append(best)

result_exp  = apply_bpe("non-disclosure", merges_exp)
result_main = apply_bpe("non-disclosure", merges)

print(f"With n_merges_exp = {n_merges_exp}:")
print(f"  'non-disclosure' tokens : {result_exp}  ({len(result_exp)} tokens)")
print(f"  BPE vocabulary size     : {len(bpe_vocab_exp)} word forms")
print()
print(f"For comparison, n_merges = {n_merges} (main run above):")
print(f"  'non-disclosure' tokens : {result_main}  ({len(result_main)} tokens)")
delta = len(result_exp) - len(result_main)
print(f"  -> {n_merges} merges achieves {abs(delta)} fewer tokens on non-disclosure vs {n_merges_exp} merges")

#### What just happened — and what's missing

BPE iteratively merged the most frequent character pair at each step. Early merges consolidate common English bigrams (`on`, `er`, `in`, `al`). Legal-specific patterns merge in later steps. The compression is real and crucially — no OOV ever, because any unseen legal compound decomposes into subwords the model has seen.

**What's missing:** building BPE from scratch on every deployment is impractical. Production systems use _pre-built_ tokenizers trained on billions of words with 50,000+ merges. That's GPT-2's tiktoken — next.

---

## Part 3 — Real BPE: GPT-2 tiktoken <a id='part-3'></a>

_The firm's question:_ "The BPE we built compresses 'non-disclosure' reasonably well — but what about 'dommages-intérêts'? Does the same algorithm handle French without us building a separate French tokenizer?"

GPT-2's tokenizer is BPE — the exact same algorithm from Part 2 — but trained on ~40GB of web text with **50,257 merge operations** instead of 50. At that scale, it has seen enough French text to merge French character sequences efficiently.

The `Ġ` prefix you'll see in the output below is GPT-2's way of marking a leading space: `Ġhello` means `' hello'` (space + hello). This is how BPE represents word boundaries without a dedicated separator token.

In [ ]:
# GPT-2 tiktoken: Import with Graceful Fallback
# Prefer the real GPT-2 BPE tokenizer; fall back gracefully if it isn't installed
try:
    import tiktoken
    enc = tiktoken.encoding_for_model("gpt2")
    TIKTOKEN_AVAILABLE = True
    print(f" tiktoken loaded  |  GPT-2 vocabulary size: {enc.n_vocab:,} tokens")
    print(f"  This is BPE with {enc.n_vocab - 256} merge operations (beyond 256 byte tokens)")
except ImportError:
    TIKTOKEN_AVAILABLE = False
    print("[tiktoken not installed — install with: pip install tiktoken]")
    print("[Showing expected output from a reference run below]")

### Predict first — GPT-2 token counts for legal phrases

GPT-2's tokenizer ran **50,257 merge operations** on ~40 GB of web text. Before you see the numbers, predict: how many tokens will it produce for `'indemnification'` (15 characters)?

| Option  | Tokens for `'indemnification'`                                            |
| ------- | ------------------------------------------------------------------------- |
| **(a)** | 1 token — seen so often in training data that it was merged completely    |
| **(b)** | ~4 tokens — broken into subword units like `in·dem·nific·ation`           |
| **(c)** | 15 tokens — so rare that GPT-2 treats every character as a separate token |

And for French `'dommages-intérêts'` (18 characters)?

| Option  | Tokens for `'dommages-intérêts'`                              |
| ------- | ------------------------------------------------------------- |
| **(a)** | 1–2 tokens — common French phrase, fully merged at 50k merges |
| **(b)** | 6–8 tokens — uncommon enough to stay partially fragmented     |
| **(c)** | 18+ tokens — accented characters cause OOV explosion          |

In [ ]:
# Token Count vs. Character Count for 10 Legal Phrases
phrases = [
    "non-disclosure agreement",
    "indemnification",
    "force majeure",
    "dommages-intérêts",
    "confidentiality",
    "non-compete covenant",
    "liquidated damages",
    "intellectual property",
    "severability clause",
    "breach of contract",
]

_ref = {
    "non-disclosure agreement": 4, "indemnification": 4, "force majeure": 3,
    "dommages-intérêts": 6, "confidentiality": 4, "non-compete covenant": 5,
    "liquidated damages": 4, "intellectual property": 4, "severability clause": 5,
    "breach of contract": 4,
}

print(f"{'Phrase':<30}  {'Tokens':>6}  {'Chars':>5}  {'Chars/Token':>11}  Notes")
print("-" * 74)

# Compare chars-per-token for each phrase, flagging French text with accented characters
for phrase in phrases:
    n_chars = len(phrase)
    n_toks  = len(enc.encode(phrase)) if TIKTOKEN_AVAILABLE else _ref.get(phrase, "?")
    if isinstance(n_toks, int):
        ratio = n_chars / n_toks
        note  = "<- French, no OOV!" if any(ord(c) > 127 for c in phrase) else ""
        print(f"  {phrase:<28}  {n_toks:>6}  {n_chars:>5}  {ratio:>10.1f}  {note}")
    else:
        print(f"  {phrase:<28}  {'?':>6}")
print()
if not TIKTOKEN_AVAILABLE:
    print("[Reference output shown — install tiktoken to see live results]")
    print("  'dommages-intérêts': 6 tokens (18 chars) — French handled, no OOV!")

In [ ]:
# The Ġ Prefix: GPT-2's Space Representation
test_sentence = "The non-disclosure agreement"
print(f"Input: {test_sentence!r}")
print()
print("Tokens with Ġ prefix  (Ġ = leading space, i.e. the start of a new word):")
print(f"{'Token ID':>10}  Decoded string")
print("-" * 40)

if TIKTOKEN_AVAILABLE:

    # Encode then decode each token individually to reveal the Ġ space-prefix marker
    tokens  = enc.encode(test_sentence)
    decoded = [enc.decode([t]) for t in tokens]
    for t, d in zip(tokens, decoded):
        print(f"  {t:>8d}  {repr(d)}")
else:
    print("  [Install tiktoken to see live output]")
    print("  Reference:")
    for tid, ds in [(464, "'The'"), (1729, "'Ġnon'"), (12, "'-'"), (15410, "'disclosure'"), (4381, "'Ġagreement'")]:
        print(f"  {tid:>8d}  {ds}")

print()
print("  -> 'The' has no Ġ because it starts the sentence (no preceding space)")
print("  -> 'Ġnon' = ' non': the space before 'non' is encoded INTO the token")
print("  -> This is how GPT-2 handles word boundaries without a [SEP] token")

In [ ]:
# Overall Compression Ratio on the Legal Corpus
total_chars = sum(len(s) for s in LEGAL_CORPUS)

# Tokenize the full corpus with GPT-2 BPE when available, else use a recorded reference count
if TIKTOKEN_AVAILABLE:
    total_tokens = sum(len(enc.encode(s)) for s in LEGAL_CORPUS)
else:
    total_tokens = 285
    print("[tiktoken not available — using reference token count for compression ratio]")

compression = total_chars / max(total_tokens, 1)
print(f"GPT-2 tokenizer compression on the legal corpus:")
print(f"  Total characters  : {total_chars:,}")
print(f"  Total GPT-2 tokens: {total_tokens:,}")
print(f"  Compression ratio : {compression:.2f} chars / token")
print()
print(f"  -> GPT-2 produces ~4-5 chars per token — roughly word-level efficiency")
print("     with zero OOV. French terms like dommages-intérêts are handled natively.")

#### What just happened — and what's missing

GPT-2's tokenizer is the algorithm from Part 2 — but run for 50,000 merges instead of 50, on billions of characters. The `Ġ` prefix is purely a representation trick: GPT-2 encodes spaces _into_ the following token rather than as a separate token, saving vocabulary slots.

**What's missing:** these token integers are just IDs — they have no semantic content. `'contract'` as token ID 2775 and `'agreement'` as token ID 4381 are as different as two random numbers. The model needs to map them to **learned vectors** where similar meanings live near each other. That's `layers.Embedding` — next.

###  Your Turn — compare our BPE vs. GPT-2 on any legal phrase

Pick any legal term from Carver & Whitmore's corpus and compare how our 50-merge scratch BPE and GPT-2's production tokenizer handle it.

**Prediction before running:** for a rare compound like `'severability'`, will GPT-2 produce fewer tokens than our toy BPE (more merges = better compression), or the same?

In [ ]:
#  Your Turn: Compare toy BPE vs. GPT-2 tiktoken on a legal phrase
my_phrase = "severability"  # <- CHANGE: try any word from the corpus

our_tokens = apply_bpe(my_phrase.lower(), merges)
print(f"Our BPE  (50 merges):  '{my_phrase}' -> {our_tokens}  ({len(our_tokens)} tokens)")

# Compare token counts only when the real GPT-2 tokenizer is available
if TIKTOKEN_AVAILABLE:
    gpt2_tokens = enc.encode(my_phrase)
    print(f"GPT-2 (50k merges):    '{my_phrase}' -> {gpt2_tokens}  ({len(gpt2_tokens)} tokens)")
    print()
    if len(our_tokens) > len(gpt2_tokens):
        print(f"  -> GPT-2 compresses better: {len(our_tokens) - len(gpt2_tokens)} fewer token(s).")
    elif len(our_tokens) == len(gpt2_tokens):
        print("  -> Same token count! Our tiny BPE matched GPT-2 on this particular word.")
    else:
        print("  -> Our toy BPE compresses better — legal corpus has higher frequency of this pattern.")
else:
    print("[Install tiktoken to see GPT-2 comparison: pip install tiktoken]")

---

## Part 4 — How a Token Row Learns <a id='part-4'></a>

_The firm's question:_ "Token IDs are just integers. How does the model learn that `'contract'` and `'agreement'` should behave similarly?"

Think of the embedding table as a **random address book**:

1. A token ID selects one row of initially random numbers.
2. Those numbers help the model predict a target token.
3. The training example reveals the correct **token ID**, not a correct vector.
4. Feedback nudges every trainable row and layer that contributed to the mistake.
5. Repeating this game gives useful structure to the table.

There is no hidden answer key saying where `'agreement'` belongs or what each dimension means. Tokens that repeatedly need to support similar predictions receive similar learning pressure, so useful neighborhoods can emerge.

The tokenizer does not participate in this learning. Its string-to-ID mapping stays fixed while the embedding table and model weights co-evolve.

### A controlled legal microscope

Carver & Whitmore's 20-sentence corpus is realistic, but too small and varied to guarantee a clean geometric lesson. For this experiment we use six deliberately paired lines:

```text
review the contract today
review the agreement today
the contract expires tomorrow
the agreement expires tomorrow
review the invoice carefully
the invoice lists charges
```

`contract` and `agreement` repeatedly need to support the same next-word predictions. `invoice` participates in different ones. That controlled difference lets us test the learning mechanism without pretending that a tiny natural corpus must produce perfect semantic clusters.

The toy model uses one word to predict the next. A production Transformer inserts many context-building layers between lookup and prediction, but the learning loop remains: **select rows → predict a token → compare with the target ID → send feedback backward**.

### Predict first — which rows should become better neighbors?

All rows begin at random positions. After repeatedly training on the six lines above, what should happen?

| Option | Prediction |
| --- | --- |
| **(a)** | `contract` and `agreement` become more similar than either is to `invoice` |
| **(b)** | `contract` and `invoice` become closest because both follow `the` once |
| **(c)** | None of the rows move because the target is an integer ID, not a vector |

Choose before running the cells below. The reveal will use both prediction loss and a plain direction-similarity score; the 2D map is only a visual aid.

In [ ]:
# Controlled Legal Corpus and Random Lookup Table
MICROSCOPE_CORPUS = [
    "review the contract today",
    "review the agreement today",
    "the contract expires tomorrow",
    "the agreement expires tomorrow",
    "review the invoice carefully",
    "the invoice lists charges",
]

# Include the main legal corpus so Part 5 can reuse this vocabulary for padding examples
_embedding_words = []
for sentence in LEGAL_CORPUS + MICROSCOPE_CORPUS:
    _embedding_words.extend(
        word
        for word in re.findall(r"[a-zA-Z][a-zA-Z-]*[a-zA-Z]|[a-zA-Z]{2,}", sentence.lower())
        if len(word) >= 3
    )

word2idx = {word: index for index, word in enumerate(sorted(set(_embedding_words)))}
idx2word = {index: word for word, index in word2idx.items()}
VOCAB_SIZE = len(word2idx)
EMBED_DIM = 16

# Capture the exact random table that training will update later
tf.random.set_seed(42)
embedding_before = layers.Embedding(VOCAB_SIZE, EMBED_DIM)
_ = embedding_before(tf.constant([0]))
embedding_before_weights = embedding_before.embeddings.numpy().copy()

FOCAL_WORDS = ["contract", "agreement", "invoice"]
print(f"Vocabulary: {VOCAB_SIZE} rows, each with {EMBED_DIM} numbers")
print("Rows to follow:")
for word in FOCAL_WORDS:
    print(f"  {word:<10} -> token ID {word2idx[word]}")
print("\n-> These rows contain random numbers. No relationship has been learned yet.")

In [ ]:
# Build Next-Word Lessons From the Controlled Lines
training_pairs = []
for sentence in MICROSCOPE_CORPUS:
    token_ids = [word2idx[word] for word in sentence.split()]
    training_pairs.extend(zip(token_ids[:-1], token_ids[1:]))

input_ids = tf.constant([input_id for input_id, _ in training_pairs], dtype=tf.int32)
target_ids = tf.constant([target_id for _, target_id in training_pairs], dtype=tf.int32)

print("Controlled next-word lessons:")
for sentence in MICROSCOPE_CORPUS:
    print(f"  {sentence}")
print(f"\n{len(training_pairs)} input -> target examples")
print("Shared pressure:")
print("  contract  -> today, expires")
print("  agreement -> today, expires")
print("Contrast:")
print("  invoice   -> carefully, lists")

In [ ]:
# Watch One Mistake Send Feedback, Then Repeat the Learning Loop
tf.random.set_seed(42)
embedding_train = layers.Embedding(VOCAB_SIZE, EMBED_DIM)
_ = embedding_train(tf.constant([0]))
embedding_train.embeddings.assign(embedding_before_weights)
loss_fn = keras.losses.SparseCategoricalCrossentropy(from_logits=True)


def predict_next(token_ids):
    """Score every vocabulary row with the same table used for input lookup."""
    selected_rows = embedding_train(token_ids)
    return tf.matmul(selected_rows, embedding_train.embeddings, transpose_b=True)


# Isolate contract -> today before training and inspect where its feedback lands
contract_id = word2idx["contract"]
today_id = word2idx["today"]
with tf.GradientTape() as probe_tape:
    probe_logits = predict_next(tf.constant([contract_id]))
    probe_loss = loss_fn(tf.constant([today_id]), probe_logits)

probe_gradient = probe_tape.gradient(probe_loss, embedding_train.embeddings).numpy()
probe_row_strength = np.linalg.norm(probe_gradient, axis=1)
strongest_rows = np.argsort(probe_row_strength)[-5:][::-1]

print("One mistake: contract -> today")
print("Strongest feedback reaches:")
for row_id in strongest_rows:
    print(f"  {idx2word[int(row_id)]:<10}  feedback strength={probe_row_strength[row_id]:.4f}")
print("-> The input row helps form the prediction; the target row helps score the answer.")
print("-> Because one table does both jobs, both routes learn together.\n")

optimizer = keras.optimizers.Adam(learning_rate=0.03)
TRAIN_STEPS = 400
losses = []

for step in range(TRAIN_STEPS):
    with tf.GradientTape() as tape:
        logits = predict_next(input_ids)
        loss = loss_fn(target_ids, logits)
    gradients = tape.gradient(loss, embedding_train.trainable_variables)
    optimizer.apply_gradients(zip(gradients, embedding_train.trainable_variables))
    losses.append(float(loss))

embedding_after_weights = embedding_train.embeddings.numpy().copy()

print(f"Training complete: {TRAIN_STEPS} feedback steps")
print(f"  Loss before: {losses[0]:.4f}")
print(f"  Loss after : {losses[-1]:.4f}")
print("  -> Repeated prediction mistakes changed the lookup rows.")
assert losses[-1] < losses[0]

In [ ]:
# One Shared 2D Map: Random Starts -> Trained Positions
# Fit one projection to both snapshots so every arrow lives in the same coordinate system
shared_projection = PCA(n_components=2)
shared_projection.fit(np.vstack([embedding_before_weights, embedding_after_weights]))
before_2d = shared_projection.transform(embedding_before_weights)
after_2d = shared_projection.transform(embedding_after_weights)

colors = {"contract": TEAL, "agreement": AMBER, "invoice": CORAL}
fig, ax = plt.subplots(figsize=(9, 6))

for word in FOCAL_WORDS:
    word_id = word2idx[word]
    start = before_2d[word_id]
    end = after_2d[word_id]
    color = colors[word]

    ax.scatter(start[0], start[1], s=110, facecolors="none", edgecolors=color, linewidths=2)
    ax.scatter(end[0], end[1], s=110, color=color)
    ax.annotate("", xy=end, xytext=start, arrowprops=dict(arrowstyle="->", color=color, lw=2))
    ax.annotate(f"{word} (after)", end, textcoords="offset points", xytext=(6, 5), color=color, fontweight="bold")

ax.set_title("Prediction Feedback Moves Token Rows\nhollow = random start, filled = after training")
ax.set_xlabel("shared map direction 1")
ax.set_ylabel("shared map direction 2")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

print("-> All arrows use one shared map; their relative movement is directly comparable.")
print("-> The map helps us see movement. The scores below test the prediction directly.")

In [ ]:
# Did Shared Prediction Pressure Make Contract and Agreement Better Neighbors?
def direction_similarity(word_1, word_2, table):
    """Return 1 for the same direction, 0 for unrelated directions, and -1 for opposites."""
    row_1 = table[word2idx[word_1]]
    row_2 = table[word2idx[word_2]]
    return float(np.dot(row_1, row_2) / (np.linalg.norm(row_1) * np.linalg.norm(row_2) + 1e-8))


contract_agreement_before = direction_similarity("contract", "agreement", embedding_before_weights)
contract_agreement_after = direction_similarity("contract", "agreement", embedding_after_weights)
contract_invoice_after = direction_similarity("contract", "invoice", embedding_after_weights)

print("Direction similarity (higher = better neighbors):")
print(f"  contract <-> agreement  before: {contract_agreement_before:+.3f}")
print(f"  contract <-> agreement  after : {contract_agreement_after:+.3f}")
print(f"  contract <-> invoice    after : {contract_invoice_after:+.3f}")
print()
print("Prediction check:")
if contract_agreement_after > contract_invoice_after:
    print("  PASS -> contract and agreement became the better-matched pair.")
else:
    print("  NOT YET -> this run did not produce the predicted ordering.")
print("  -> The model learned this from shared prediction jobs, not from a synonym label.")
assert contract_agreement_after > contract_invoice_after

#### What just happened — and what's missing

The model was never shown a correct vector for `'contract'`. It only practiced token-prediction examples. Because `'contract'` and `'agreement'` repeatedly did the same jobs, their rows received similar learning pressure and became better neighbors.

| Representation | Intuition |
| --- | --- |
| **Lookup row** | The same context-free starting row is selected whenever a token ID appears |
| **Contextual token vector** | A Transformer reshapes that starting information for this occurrence, so `'contract'` can differ in `contract expires` and `contract was signed` |

This notebook built the lookup-row mechanism. The Transformer chapter freezes a hand-drawn input map to inspect attention, then trains lookup rows and attention together in its decoder model.

Later retrieval models pool contextual token vectors into one sentence vector and refine it with positive and negative text pairs. That is a forward pointer, not another mechanism we need to build here.

### Your turn — change the learning pressure

Make `invoice` do the same jobs as `contract` and `agreement`:

```text
review the invoice today
the invoice expires tomorrow
```

Replace its two lines in `MICROSCOPE_CORPUS`, then rerun Part 4. Predict first: should `invoice` become a closer neighbor of `contract` than it is now?

In [ ]:
# Your Turn: Inspect the Focal Pair After Changing the Corpus Above
comparison_word = "invoice"  # <- CHANGE after editing MICROSCOPE_CORPUS and rerunning Part 4

comparison_score = direction_similarity("contract", comparison_word, embedding_after_weights)
reference_score = direction_similarity("contract", "agreement", embedding_after_weights)

print(f"contract <-> agreement : {reference_score:+.3f}")
print(f"contract <-> {comparison_word:<9}: {comparison_score:+.3f}")
print("\n-> Similar training jobs should make the comparison score rise.")
print("-> The data defines the pressure; the model discovers coordinates that satisfy it.")

---

## Part 5 — Padding and Masking <a id='part-5'></a>

_The firm's question:_ "Our contracts range from 1-word clauses to 500-word paragraphs. How do we feed a batch of different-length inputs to the model without the short ones distorting the loss?"

> **Concrete example:** In the three-sentence batch below — lengths [6, 1, 13] padded to 13 — 12 of 39 total label positions are padding. Without masking, those 12 positions inject misleading gradients into every parameter update.

GPUs process batches in parallel, but parallel processing requires all sequences in a batch to have the same length. The standard solution is **padding**: fill short sequences with a special `PAD` token. If the loss counts those positions as real targets, it learns from bookkeeping rather than language.

> **Framework difference:** PyTorch's `CrossEntropyLoss(ignore_index=-100)` can exclude sentinel labels directly. Keras sparse cross-entropy still requires valid class IDs, so replace padded labels with a safe ID, compute unreduced per-token loss, multiply by a real-token mask, and normalize by the mask sum. The comparison after the executable example shows both forms side by side.

In [ ]:
# Part 5: Variable-Length Sentences from the Legal Corpus
sentences_5 = [
    "The contract specifies force majeure provisions.",
    "Indemnification.",
    "All intellectual property rights are assigned to the company upon signing the agreement.",
]

PAD_ID = VOCAB_SIZE  # Padding token ID: out-of-vocabulary range


def tokenize_sent(sent, w2i, pad_id=0):
    """Tokenize a sentence into word indices, using pad_id for unknown words."""
    toks = []
    for w in re.findall(r"[a-zA-Z][a-zA-Z-]*[a-zA-Z]|[a-zA-Z]{2,}", sent.lower()):
        if len(w) >= 3:
            toks.append(w2i.get(w, pad_id))
    return toks


# Tokenize each sentence and record its raw (unpadded) length
seqs    = [tokenize_sent(s, word2idx, pad_id=0) for s in sentences_5]
max_len = max(len(s) for s in seqs)

# Right-pad every sequence to the batch's longest length with PAD_ID
padded_seqs  = [s + [PAD_ID] * (max_len - len(s)) for s in seqs]
batch_tensor = np.array(padded_seqs, dtype="int32")  # (3, max_len)

print("Variable-length sentences padded to a batch tensor:")
for i, (sent, seq) in enumerate(zip(sentences_5, seqs)):
    print(f"  [{i}] len={len(seq):>2}  {sent[:55]}")
print()
print(f"batch_tensor shape: {batch_tensor.shape}  # (n_sentences, max_len)")
print(f"PAD_ID = {PAD_ID}")
n_real = sum(len(s) for s in seqs)
n_pad  = sum(max_len - len(s) for s in seqs)
print(f"  {n_real} real tokens + {n_pad} padding tokens = {n_real + n_pad} total positions")
print(f"  {n_pad / (n_real + n_pad) * 100:.0f}% of positions are padding")

### Predict first — how much does padding inflate the loss?

The batch above has **3 sentences** padded to the same length. Some fraction of label positions are padding. If `SparseCategoricalCrossentropy` counts those positions anyway, what happens?

| Option  | Effect on the loss                                                                           |
| ------- | -------------------------------------------------------------------------------------------- |
| **(a)** | Loss is slightly higher (≈ +0.05) — padding adds a very small but negligible bias            |
| **(b)** | Loss is noticeably higher (≈ +0.4 or more) — the many pad positions inflate it significantly |
| **(c)** | Loss is identical — the model quickly learns to predict PAD and those positions wash out     |

And what happens to **gradients** for padding positions without masking?

| Option  | Gradient effect                                                                |
| ------- | ------------------------------------------------------------------------------ |
| **(a)** | No gradient flows — Keras automatically ignores PAD tokens                     |
| **(b)** | Gradients flow, pushing model weights toward predicting PAD after real content |
| **(c)** | Gradients are small but cancel each other out                                  |

In [ ]:
# Without masking: Padding Inflates the Loss (WRONG)
tf.random.set_seed(42)
embed_pad  = layers.Embedding(VOCAB_SIZE + 1, EMBED_DIM)
linear_pad = layers.Dense(VOCAB_SIZE + 1, use_bias=False)

inputs  = tf.constant(batch_tensor[:, :-1], dtype=tf.int32)   # (3, max_len-1)
targets = tf.constant(batch_tensor[:, 1:],  dtype=tf.int32)   # (3, max_len-1)

logits_pad = linear_pad(embed_pad(inputs))   # (3, max_len-1, VOCAB_SIZE+1)

loss_fn_no_mask = keras.losses.SparseCategoricalCrossentropy(from_logits=True, reduction='sum_over_batch_size')
loss_no_mask    = loss_fn_no_mask(targets, logits_pad)

print("Without masking:  padding positions counted as real predictions (WRONG)")
print(f"  Loss = {float(loss_no_mask):.4f}")
print()
print("  Problem:")
print(f"    Total prediction positions : {int(np.prod(targets.shape))}")
print(f"    Real token positions       : {int((targets.numpy() != PAD_ID).sum())}")
print(f"    Padding positions          : {int((targets.numpy() == PAD_ID).sum())}")
print()
print("  The model is penalized for wrong predictions on PAD tokens — but PAD")
print("  is not a real word. The loss is inflated and the gradients are misleading.")

In [ ]:
# With masking: Only Real Tokens Contribute (CORRECT)
# Strategy: create a sample_weight mask (1 for real tokens, 0 for padding)
# and use SparseCategoricalCrossentropy with the mask.
# Convention mirrors PyTorch's ignore_index=-100: set pad positions to -1
# and compute loss only where labels != PAD_ID.

labels_masked = targets.numpy().copy()
mask          = (labels_masked != PAD_ID).astype("float32")   # (3, max_len-1)

# Replace PAD positions with 0 (they'll be masked out by sample_weight)
labels_masked[labels_masked == PAD_ID] = 0

# Use sample_weight to zero-out padding contributions
loss_fn_masked = keras.losses.SparseCategoricalCrossentropy(from_logits=True, reduction='none')
per_token_loss = loss_fn_masked(labels_masked, logits_pad)   # (3, max_len-1)  EagerTensor

# Convert to numpy before element-wise multiply and sum (TF tensors don't support .sum())
loss_masked    = float((per_token_loss.numpy() * mask).sum() / mask.sum())

n_real_contributing = int(mask.sum())

print("With masking (sample_weight):  only real tokens contribute to the loss (CORRECT)")
print(f"  Loss = {loss_masked:.4f}")
print()
print(f"   Only {n_real_contributing} real tokens contribute to loss")
print(f"  Gradient flows only from actual contract language — not from padding.")
print()
print("  Keras equivalent of PyTorch's ignore_index=-100:")
print("    labels[labels == pad_token_id] = 0  # neutralize")
print("    mask = (original_labels != pad_token_id).astype('float32')")
print("    loss = (per_token_loss.numpy() * mask).sum() / mask.sum()")
print()
diff = float(loss_no_mask) - loss_masked
print(f"  Loss difference: {diff:+.4f}")
if diff > 0:
    print("  -> Without masking, loss is higher — padding positions inflate it")
    print("     and gradients point partly toward nonsense predictions.")
else:
    print("  -> With untrained random weights the raw loss magnitude isn't the tell —")
    print("     what matters is that masking guarantees only the real tokens drive the")
    print("     gradient, instead of padding positions pulling weights in noisy directions.")

#### TensorFlow/Keras ↔ PyTorch: ignore padding in the loss

The attention mask and the loss mask solve different problems. Here we compare only the loss rule: padded target positions must contribute neither loss nor gradient.

| TensorFlow/Keras | PyTorch |
|---|---|
| ```python
safe_labels = tf.where(real_token_mask, labels, 0)
per_token = keras.losses.sparse_categorical_crossentropy(
    safe_labels, logits, from_logits=True
)
mask = tf.cast(real_token_mask, per_token.dtype)
loss_tf = tf.reduce_sum(per_token * mask) / tf.reduce_sum(mask)
``` | ```python
labels_pt = labels.clone()
labels_pt[~real_token_mask] = -100
loss_pt = F.cross_entropy(
    logits.transpose(1, 2),
    labels_pt,
    ignore_index=-100,
)
``` |

**Invariant:** only real target tokens contribute. Keras requires valid class IDs followed by explicit masking; PyTorch can encode the same exclusion directly with `ignore_index`. Neither mechanism replaces the attention mask that controls which input positions a sequence model may read.

In [ ]:
# Visualize: Padded Batch Tensor + Attention Mask
import matplotlib.colors as mcolors

# Rebuild the padded batch as a plain array and derive its boolean real/pad mask
batch_np = np.array(padded_seqs)
mask_np  = (batch_np != PAD_ID).astype(float)

fig5, (ax5a, ax5b) = plt.subplots(1, 2, figsize=(14, 3))

# Left panel: heatmap of token IDs (red = padding, green = real token)
cmap_tok = plt.get_cmap("RdYlGn_r").copy()
im_tok   = ax5a.imshow(batch_np, aspect="auto", cmap=cmap_tok, vmin=0, vmax=VOCAB_SIZE)
ax5a.set_title("Padded Batch  (green = real token, red = padding)", fontsize=10)
ax5a.set_xlabel("Token position"); ax5a.set_ylabel("Sentence")
ax5a.set_yticks([0, 1, 2])
ax5a.set_yticklabels(["short", "v.short", "long"], fontsize=8)
plt.colorbar(im_tok, ax=ax5a, label="Token ID")

# Right panel: the binary mask marking which positions should count toward loss
cmap_mask = plt.get_cmap("RdYlGn").copy()
im_mask   = ax5b.imshow(mask_np, aspect="auto", cmap=cmap_mask, vmin=0, vmax=1)
ax5b.set_title("Attention Mask  (1 = real, 0 = padding/ignore)", fontsize=10)
ax5b.set_xlabel("Token position")
ax5b.set_yticks([0, 1, 2])
ax5b.set_yticklabels(["short", "v.short", "long"], fontsize=8)
plt.colorbar(im_mask, ax=ax5b, label="Mask value")

plt.tight_layout()
plt.show()

print(f"  -> Short sentence (row 1) has {max_len - len(seqs[1])} padding positions at the right end")
print("  -> The mask tells the model: only compute loss over the green positions")

#### What just happened — and what's missing

Padding makes variable-length batches possible. The fix is a masking pattern:

1. `mask = (labels != pad_token_id).astype('float32')` — identify real token positions
2. `loss = (per_token_loss * mask).sum() / mask.sum()` — weight loss by mask

This is the Keras equivalent of PyTorch's `CrossEntropyLoss(ignore_index=-100)`. The attention mask (the right panel above) is the inference-time equivalent, telling self-attention layers not to attend to padding positions.

**What's missing:** the vocabulary in this notebook is ~150 words. GPT-2 uses 50,257. LLaMA-3 uses 128,000. The mechanics are identical — just wider matrices. The final Part makes that comparison explicit.

###  Your Turn — vary the padding fraction and measure the loss gap

**What happens to the loss gap (no masking vs. with masking) as the fraction of padding increases?**

**Prediction before running:** if you replace the medium sentence with a one-word sentence, making the batch ~80% padding, will the loss gap grow, shrink, or stay the same? Change `sentences_your_turn` below.

In [ ]:
#  Your Turn: Vary padding fraction and measure loss gap
sentences_your_turn = [
    "Indemnification.",  # <- CHANGE: try shorter/longer sentences
    "The contract.",
    "All intellectual property rights are assigned to the company upon signing the agreement.",
]

tf.random.set_seed(42)

# Re-tokenize and pad this new batch exactly as in the earlier example
seqs_yt      = [tokenize_sent(s, word2idx, pad_id=0) for s in sentences_your_turn]
max_len_yt   = max(len(s) for s in seqs_yt)
padded_yt    = [s + [PAD_ID] * (max_len_yt - len(s)) for s in seqs_yt]
batch_yt     = np.array(padded_yt, dtype="int32")

n_real_yt = sum(len(s) for s in seqs_yt)
n_pad_yt  = sum(max_len_yt - len(s) for s in seqs_yt)
pad_pct   = n_pad_yt / (n_real_yt + n_pad_yt) * 100

embed_yt  = layers.Embedding(VOCAB_SIZE + 1, EMBED_DIM)
linear_yt = layers.Dense(VOCAB_SIZE + 1, use_bias=False)

inputs_yt  = tf.constant(batch_yt[:, :-1], dtype=tf.int32)
targets_yt = batch_yt[:, 1:]
logits_yt  = linear_yt(embed_yt(inputs_yt))

# Rebuild the same mask-and-neutralize steps used above for this new batch
mask_yt    = (targets_yt != PAD_ID).astype("float32")
targets_masked_yt = targets_yt.copy()
targets_masked_yt[targets_masked_yt == PAD_ID] = 0

# Compute loss with and without masking to measure the gap at this padding fraction
loss_no_m = float(loss_fn_no_mask(tf.constant(targets_yt), logits_yt))
ptl_yt    = float((loss_fn_masked(tf.constant(targets_masked_yt), logits_yt).numpy() * mask_yt).sum() / max(mask_yt.sum(), 1))

print(f"Batch padding fraction: {pad_pct:.0f}%  ({n_pad_yt} pad / {n_real_yt + n_pad_yt} total positions)")
print(f"  Loss WITHOUT masking : {loss_no_m:.4f}")
print(f"  Loss WITH    masking : {ptl_yt:.4f}")
print(f"  Gap                  : {loss_no_m - ptl_yt:+.4f}")
print()
print("  -> Higher padding fraction = larger gap. More pad positions = more misleading gradients.")
print("     For Carver & Whitmore: 1-word clauses next to 500-word paragraphs can reach 90% padding.")

---

## Part 6 — Toy → Real Bridge <a id='part-6'></a>

_The firm's question:_ "Everything you've built here uses ~150 words and 16-dimensional vectors. GPT-2 uses 50,000 tokens. Is the architecture literally the same?"

Yes — same `layers.Embedding`, same BPE algorithm, same masking training convention. The only difference is scale: vocabulary size, embedding dimension, and consequently the size of $W_e$.

![Tokenization pipeline: raw string -> BPE tokens -> integer IDs -> embedding vectors -> model input](images/tokenization-pipeline.png)

In [ ]:
# Toy -> Real Bridge: Scale Comparison
print("=" * 72)
print(f"{'Component':<22}  {'This notebook':>14}  {'GPT-2 (124M)':>13}  {'LLaMA-3-8B':>14}")
print("=" * 72)

rows = [
    ("Vocabulary rows",       f"~{VOCAB_SIZE} words",       "50,257",           "128,000"),
    ("Numbers per row",       f"{EMBED_DIM}",               "768",              "4,096"),
    ("Lookup parameters",     f"{VOCAB_SIZE*EMBED_DIM:,}",  "38,597,376",       "524,288,000"),
    ("Tokenizer",             "word-level",                 "byte-level BPE",   "large subword vocab"),
    ("Context length",        "~20 words",                  "1,024",            "8,192"),
    ("Padding loss",          "weighted mask",              "ignored labels",   "ignored labels"),
]
for row in rows:
    print(f"  {row[0]:<20}  {row[1]:>14}  {row[2]:>13}  {row[3]:>14}")

print("=" * 72)
print()
print("  -> The lookup operation stays simple at every scale: token ID selects one row.")
print("  -> Production models use far more rows and far wider rows, so the table becomes expensive.")
print("  -> Their tokenizers and surrounding Transformer architectures also differ.")
print("  -> GPT-2 lookup parameters alone: 38.6M (about 31% of its 124M parameters)")

In [ ]:
# GPT-2 Tokenizer Demo (requires transformers)
# Try the real GPT-2 tokenizer; fall back to recorded reference output if unavailable
try:
    from transformers import GPT2Tokenizer
    _gpt2_tok = GPT2Tokenizer.from_pretrained("gpt2")
    print(f"GPT-2 vocabulary size: {_gpt2_tok.vocab_size:,}")
    print()
    for phrase in ["the cat", "non-disclosure", "indemnification", "dommages-intérêts", "force majeure"]:
        toks = _gpt2_tok.encode(phrase)
        print(f"  {phrase!r:<28} -> {toks}  ({len(toks)} tokens)")
except Exception as _e:
    print("[transformers not installed or GPT-2 model not cached]")
    print("[Showing reference output from a production run]")
    print()
    _ref_gpt2 = [
        ("'the cat'",           "[1169, 3797]",           "2 tokens"),
        ("'non-disclosure'",    "[3642, 12, 15410, 495]", "4 tokens"),
        ("'indemnification'",   "[521, 1516, 6637, 341]", "4 tokens"),
        ("'dommages-intérêts'", "[67, 5908, 363, 12, ...]","6 tokens"),
        ("'force majeure'",     "[3174, 2233, 2850]",     "3 tokens"),
    ]
    for phrase, toks, count in _ref_gpt2:
        print(f"  {phrase:<28} -> {toks}  ({count})")

print()
print("  -> 'non-disclosure' splits at the hyphen: 4 tokens — same insight as our scratch BPE")
print("  -> 'dommages-intérêts' handled natively — no OOV despite being French!")

In [ ]:
# Closing Decision: Recommendation for the Law Firm
print("Law Firm Tokenization Recommendation")
print("=" * 52)
print()

# Base the recommendation on measured compression when tiktoken is available
if TIKTOKEN_AVAILABLE:
    ratio = total_chars / total_tokens
    print(f"Measured on the 20-sentence legal corpus:")
    print(f"  GPT-2 tiktoken compression: {ratio:.2f} chars / token")
    print()
    if ratio > 4.0:
        print("   GPT-2 tiktoken is RECOMMENDED for the law firm.")
        print("    -> Handles 'indemnification' in ~4 tokens (never OOV)")
        print("    -> Handles French 'dommages-intérêts' without a separate vocabulary")
        print(f"    -> {ratio:.1f} chars/token: near word-level efficiency with zero OOV")
    else:
        print(f"  Warning: Compression ratio ({ratio:.1f}) is lower than typical English.")
        print("    -> Consider a domain-specific tokenizer fine-tuned on legal corpora.")
else:
    print("[tiktoken not available — install to see actual compression ratio]")
    print()
    print("Reference: GPT-2 tiktoken typically achieves 4.5-5.5 chars/token on English legal text.")
    print()
    print("   GPT-2 tiktoken is RECOMMENDED for the law firm.")
    print("    -> 'indemnification' -> ~4 tokens (not OOV)")
    print("    -> 'dommages-intérêts' -> ~6 tokens (French handled natively)")

#### What just happened — and what's missing

The toy-to-real bridge confirms two things:

1. **The lookup idea stays simple:** a token ID selects one trainable row.
2. **Scale changes cost:** GPT-2 stores 50,257 rows with 768 numbers each — 38.6 million lookup parameters before counting the rest of the model.

Production tokenizers, context limits, and Transformer architectures differ; the table lookup is the shared primitive.

**What's still missing:** a lookup row is context-independent. `'contract'` starts from the same row in `contract expires` and `contract was signed`. Self-attention builds a different contextual vector for each occurrence.

### Forward pointers

The legal microscope used one word to predict the next. Riverside's later Transformer adds context-building layers between lookup and prediction, but learns through the same feedback loop.

#### Animated training loop

![Animation tracing fixed tokenization, trainable lookup rows, Transformer context building, vocabulary prediction, backward feedback, and repeated improvement.](images/embedding-transformer-training-loop.gif)

Watch the direction change halfway through: information first moves **forward** from IDs to probabilities, then prediction error travels **backward** into the lookup rows and Transformer weights. Repeating that cycle raises the target token's probability and lowers the loss.

#### Detailed static reference

![A masked Riverside sentence flows through a fixed tokenizer, random lookup rows, contextual processing, vocabulary prediction, backward feedback, and repeated updates that make the target token more likely.](images/how-a-token-learns-where-to-live.jpg)

> **-> `../../genai/01-transformers/01-attention-and-transformer-blocks.ipynb`**: Part 1 deliberately freezes a hand-authored 3D map so you can inspect what attention does without also moving the input space. Those named axes are a teaching scaffold, not learned dimensions.

> **-> `../../genai/01-transformers/02-decoder-only-language-model.ipynb`**: The MiniLM replaces that scaffold with a trainable `nn.Embedding`. Its lookup rows and attention weights co-evolve under next-token loss, and weight tying reuses the same table to score output tokens.

> **-> `../05-rnn-sequence-modeling`**: The RNN notebook uses `layers.Embedding` to predict the next character in a melody sequence. It uses the same token-ID → lookup-row → prediction-error → backward-update contract built in Part 4.

---

## Summary <a id='summary'></a>

### Completed Roadmap

| Part | Concept | What we proved |
|------|---------|----------------|
| 1 | Why tokenization exists | ~60 unique chars (zero OOV, 5× longer sequences) vs. ~340 words (compact, 200 OOV risk words) |
| 2 | BPE from scratch | Merge-pair algorithm built from `get_pairs` + `merge_vocab`; `'non-disclosure'` shrinks from 14 chars to a small number of subwords after 50 merges |
| 3 | Real BPE: GPT-2 tiktoken | Leading spaces in decoded token fragments preserve word boundaries; French `'dommages-intérêts'` is split into a compact subword sequence |
| 4 | How a token row learns | A controlled legal corpus showed prediction loss falling, input and target rows receiving feedback, and words with shared prediction jobs becoming better neighbors |
| 5 | Padding and masking | `mask = (labels != pad_id).astype('float32')` + weighted loss ensures only real tokens contribute; attention mask for inference-time |
| 6 | Toy → real bridge | 16-dim / 150 vocab → 768-dim / 50k vocab (GPT-2) → 4096-dim / 128k vocab (LLaMA-3): same lookup principle, larger learned table |

### Key Insights to Keep

- **BPE is not magic** — it is a greedy merge loop. The tokenizer creates stable IDs; gradient descent does not update it.
- **Whitespace is encoded inside token byte sequences.** `tiktoken.decode([token_id])` displays a leading space when that token begins at a word boundary; the `Ġ` glyph is a visualization used by some other byte-level BPE tokenizers, not this API's decoded output.
- **There is no correct embedding vector in the training data.** The label is the correct token ID; useful geometry emerges while trainable rows and model layers reduce prediction error together.
- **A lookup row is context-independent.** The same token ID retrieves the same starting row; Transformer attention produces a new contextual vector for each occurrence.
- **Embedding dimensions are distributed features.** Learned axes are not individually assigned meanings such as "legal" or "person."
- **Padding masking is a contract**, not an optimization. Every fine-tuning pipeline uses it.

---

## When to Use What — Tokenization Decisions

| Situation                                    | Choice                                                  | Reason                                                                        |
| --------------------------------------------- | ------------------------------------------------------- | ----------------------------------------------------------------------------- |
| General-purpose LLM (English, code)          | GPT-2 / tiktoken BPE (50k vocab)                        | Best balance of compression and OOV handling; standard for open-source LLMs   |
| Multilingual LLM                             | SentencePiece Unigram (100k+ vocab)                     | Byte-level BPE or Unigram handles non-Latin scripts without OOV explosions    |
| Legal, medical, or domain-specific corpus    | Domain-adapted BPE (retrained tokenizer)                | Standard tokenizer over-fragments rare compound terms                          |
| Padding variable-length sequences in a batch | `pad_token_id` + `attention_mask` + loss masking        | Always mask pad positions; never let loss propagate through padding            |
| Embedding a small vocabulary (toy model)     | `layers.Embedding(vocab_size, d_model)`                 | Trainable lookup table; access weights via `.embeddings`                       |

→ **Next:** [`../07-pytorch-rnn-bridge/01-pytorch-rnn-bridge.ipynb`](../07-pytorch-rnn-bridge/01-pytorch-rnn-bridge.ipynb) — carry these token, embedding, padding, and loss contracts into a PyTorch sequence model before replacing recurrence with attention.

---

## Tier 1 / 2 / 3 Ledger

**Tier 1 — built from scratch, measured, proven:** character-level tokenization, word-level tokenization with OOV analysis, BPE merge algorithm, GPT-2 tiktoken, `layers.Embedding`, padding + masking.

**Tier 2 — same algorithm, different scoring (explained, not built):** WordPiece (BERT's tokenizer) uses the same merge-pair structure as BPE but scores pairs by likelihood ratio rather than raw frequency.

**Tier 3 — named, one-line rationale each (not built):**

- _SentencePiece_: language-agnostic BPE/Unigram that treats whitespace as a normal character (useful for Japanese/Chinese).
- _Unigram Language Model_: maintains a probabilistic vocabulary and prunes it; used in XLNet, ALBERT.
- _Byte-level BPE_: operates on raw UTF-8 bytes, giving a truly universal vocabulary with zero OOV (used in GPT-3, GPT-4, Falcon).